## 1. Import the libraries:

In [53]:
import pandas as pd
import os

from src.customer_churn.logging.logger import logging

In [51]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

In [7]:
os.chdir("../")
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 2. Import the training data:

In [8]:
train = pd.read_parquet("Artifacts/data_cleaning/data_clipped/train.parquet")

In [9]:
train.head(3)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,orders_returned,orders_cancelled,quantity_abs,quantity_abs_clipped,price_clipped
0,546782,22170,PICTURE FRAME WOOD TRIPLE PORTRAIT,4,2011-03-17 09:13:00,6.75,15291.0,United Kingdom,0,0,4,4,6.75
1,535748,21361,LOVE LARGE WOOD LETTERS,1,2010-11-28 13:21:00,12.75,14508.0,United Kingdom,0,0,1,1,12.75
2,490917,21527,RETRO SPOT TRADITIONAL TEAPOT,4,2009-12-08 13:14:00,7.95,15162.0,United Kingdom,0,0,4,4,7.95


# 3. Define the churn label:

- The transactions after the date `2011-03-31` are already dropped during data ingestion.
- The observation window would be `2009-12-01`(this is the first purchase invoice date recorded in the dataset) till `2010-12-31` + filter the customer out if the purchase quantity is just 1
- The churn window would be **3 months + the observation window** i.e. `2011-01-01` to `2011-03-31`

#### Logic implemented for defining the churn label:
- The churn window will be used as a filter to define the churn(1) or retained(0)
- If customer didn't make a single purchase in **the churn window** then 1(churn) else 0(retained)

#### Steps involved:
1. The `dataframe` will be split into two dataframes: `df with observation window(observation_df)` and `df with churn window(churn_df)`
2. Get the unique `customerid` from `churn_df`
3. Create a new feature `churn` on `observation_df`, if `customerid` is in `customerid` of `churn_df` then 0 (retained) else 1(churn).

## 3.1 Add the churn feature:

In [54]:
churn_cutoff_date = '2011-03-31'
chun_cutoff_date = pd.to_datetime(churn_cutoff_date)
observation_date = chun_cutoff_date - pd.DateOffset(months=3)
observation_date

Timestamp('2010-12-31 00:00:00')

In [57]:
observation_date

Timestamp('2010-12-31 00:00:00')

In [39]:
chun_cutoff_date

Timestamp('2011-03-31 00:00:00')

In [40]:
observation_df = train[train['invoicedate'] <= observation_date].copy()
churn_df = train[(train['invoicedate'] > observation_date) & (train['invoicedate'] <= chun_cutoff_date)].copy()

In [41]:
observation_df.shape

(341428, 13)

In [42]:
observation_df.shape

(341428, 13)

In [43]:
churn_df.shape

(54741, 13)

In [44]:
print(f"The min invoice date in observation df: {observation_df['invoicedate'].min()} and the max is: {observation_df['invoicedate'].max()}")
print(f"{'-' * 25}")

customer_ids_from_observation_df = set(observation_df['customerid'].unique())
print(f"There are a total of: {len(customer_ids_from_observation_df)} customer IDs who made the transaction in observation window.")

The min invoice date in observation df: 2009-12-01 07:45:00 and the max is: 2010-12-23 16:06:00
-------------------------
There are a total of: 4379 customer IDs who made the transaction in observation window.


In [45]:
print(f"The min invoice date in churn df: {churn_df['invoicedate'].min()} and the max is: {churn_df['invoicedate'].max()}")

The min invoice date in churn df: 2011-01-04 10:00:00 and the max is: 2011-03-30 17:33:00


In [46]:
customer_ids_from_churn_df = set(churn_df['customerid'].unique())
print(f"There are a total of: {len(customer_ids_from_churn_df)} customer IDs who made the transaction in churn window.")

There are a total of: 1774 customer IDs who made the transaction in churn window.


In [47]:
observation_df['churn'] = (~observation_df['customerid'].isin(customer_ids_from_churn_df)).astype(int)

In [48]:
observation_df['churn'].value_counts()

churn
0    204698
1    136730
Name: count, dtype: int64

## 3.2 Transform into function:

In [59]:
def define_churn_label(df:pd.DataFrame, churn_cutoff_date:str, data_name:str) -> pd.DataFrame:
    """
    Adds a new feature churn label: 1(Churn) & 0(Retain)
    Parameters:
    -----------
    df: DataFrame where churn feature needs to be added
    churn_cutoff_date: The churn window end date
    """
    # Set the observation and churn date:
    chun_cutoff_date = pd.to_datetime(churn_cutoff_date)
    observation_date = chun_cutoff_date - pd.DateOffset(months=3)
    logging.info(f"Churn window frame from {observation_date + pd.DateOffset(days=1)} : {churn_cutoff_date}")
    logging.info(f"Observation window is set till: {observation_date}")    

    # Split the data into observation and churn window dataframe:
    observation_df = df[df['invoicedate'] <= observation_date].copy()
    churn_df = train[(df['invoicedate'] > observation_date) & (df['invoicedate'] <= chun_cutoff_date)].copy()
    customer_ids_from_observation_df = set(observation_df['customerid'].unique())

    # Logs:
    logging.info(f"Observation df of shape: ({observation_df.shape}) and churn df of shape: ({churn_df.shape}) created.")
    logging.info(f"Observation df has invoice date of min : {observation_df['invoicedate'].min()} and max:       {observation_df['invoicedate'].max()} | Churn df has invoice date of min : {churn_df['invoicedate'].min()} and max:{churn_df['invoicedate'].max()}")

    # Labelling churn:
    observation_df['churn'] = (~observation_df['customerid'].isin(customer_ids_from_churn_df)).astype(int)
    
    # Logs:
    logging.info(f"churn feature created | The count of churn(1) and retain(0) is:\n{observation_df['churn'].value_counts()}")
    
    return df
    
    

# 4. Start Creating New Features:

### 4.1 Adding `total_price` column:

In [64]:
observation_df.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,orders_returned,orders_cancelled,quantity_abs,quantity_abs_clipped,price_clipped,churn
1,535748,21361,LOVE LARGE WOOD LETTERS,1,2010-11-28 13:21:00,12.75,14508.0,United Kingdom,0,0,1,1,12.75,1
2,490917,21527,RETRO SPOT TRADITIONAL TEAPOT,4,2009-12-08 13:14:00,7.95,15162.0,United Kingdom,0,0,4,4,7.95,1


In [65]:
observation_df['total_amount'] = observation_df['quantity'] * observation_df['price']

In [66]:
observation_df.head(3)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,orders_returned,orders_cancelled,quantity_abs,quantity_abs_clipped,price_clipped,churn,total_amount
1,535748,21361,LOVE LARGE WOOD LETTERS,1,2010-11-28 13:21:00,12.75,14508.0,United Kingdom,0,0,1,1,12.75,1,12.75
2,490917,21527,RETRO SPOT TRADITIONAL TEAPOT,4,2009-12-08 13:14:00,7.95,15162.0,United Kingdom,0,0,4,4,7.95,1,31.80
3,504902,21906,PHARMACIE FIRST AID TIN,2,2010-04-18 15:35:00,6.75,13286.0,United Kingdom,0,0,2,2,6.75,1,13.50
